# GENERATE EXOME BURDEN SCORE

In [ ]:
# Load packages 
import re
import os
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce

# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
# Remove previously generated polygenic risk score (PRS) files
!rm -rf *PRS.tsv

# Remove any existing gene list directory and recreate it
!rm -rf gene_list
!mkdir -p gene_list 

# Create a directory to store the downloaded gene burden sum scores
!mkdir -p burden_sum_scores

In [ ]:
%%bash 
# Define the Google Cloud Storage bucket containing the exome gene burden sum scores
my_bucket="gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe"

# Create a directory to store the chromosome-specific exome burden score files
  mkdir -p burden_scores/burden_scores_exome
  
# Iterate over autosomes 1–22 to download the corresponding exome burden score files
for chrom in {1..22}; do

  # Download the exome gene burden score file for the current chromosome
  # Files are retrieved from the EUR exome burden score directory in the workspace bucket
  # and saved locally in the exome-specific burden score directory
  gsutil cp "${my_bucket}/data/gene_burden_analysis/burden_sum_scores/exome_burden_sum_scores/EUR-Exome-burden-prs-chr${chrom}.sscore" burden_scores/burden_scores_exome
  
done

In [ ]:
# Define the directory containing the exome gene burden score files
exome_folder = "burden_scores/burden_scores_exome"

# Identify all gene burden score files in the exome directory
# Sort filenames to ensure consistent processing order
exome_filenames = sorted(glob.glob(os.path.join(exome_folder, "*")))

# Initialize a list to store the filtered gene burden score dataframes
exome_file = None 
exome_files = []

# Load individual gene burden score files and append valid files to the list
for exome_filename in exome_filenames: 
    # Load the gene-specific burden score file
    exome_file = pd.read_csv(exome_filename, sep='\t')
    
    # Verify that the sum of named allele dosages matches the reported gene burden score
    # Exclude files that do not satisfy this consistency check
    if not (exome_file["NAMED_ALLELE_DOSAGE_SUM"] == exome_file["SCORE1_SUM"]).all():
        continue
    
    # Retain only the participant identifier and corresponding gene burden score
    exome_file = exome_file[['IID','SCORE1_SUM']]
    
    # Append the filtered gene burden scores to the list
    exome_files.append(exome_file) 

# Assign a unique column identifier to each gene-specific burden score
# This ensures that gene burden scores can be distinguished after merging
for i, exome_file in enumerate(exome_files):
    
    # Retain the participant identifier and gene burden score
    exome_file = exome_file[['IID', 'SCORE1_SUM']].copy()
    
    # Rename the gene burden score column using a unique numerical identifier
    exome_file = exome_file.rename(columns={'SCORE1_SUM': f'SCORE1_SUM_{i}'})
    
    # Replace the original dataframe with the renamed dataframe
    exome_files[i] = exome_file

# Perform an inner join across all gene-specific dataframes using IID
# Only participants present in every gene burden file will be retained
exome_files_merged = reduce(lambda left, right: pd.merge(left, right, on='IID', how='inner'), exome_files)

# Calculate the aggregate exome gene burden score for each participant
# Identify all columns containing individual gene burden scores
numeric_cols = [col for col in exome_files_merged.columns if col.startswith('SCORE1_SUM')]

# Sum the individual gene burden scores across all genes
# to generate a single aggregate exome burden score per participant
exome_files_merged['EXOME_SCORE1_SUM'] = exome_files_merged[numeric_cols].sum(axis=1)

# Export the aggregated exome gene burden scores as a tab-delimited file
# Each row represents a participant and EXOME_SCORE1_SUM represents
# the sum of their gene-specific exome burden scores
exome_files_merged.to_csv(f"exome-burden-PRS.tsv", sep ='\t', index=False)

In [ ]:
# Upload exome burden prs to workspace bucket
!gsutil cp exome-burden-PRS.tsv {my_bucket}/data/gene_burden_analysis/burden_gene_sets/exome/ 